# 04 — O5 : recommandations

Ce notebook **visualise** les données qui alimentent les indicateurs de O5. Il ne produit aucun chiffre du 04 : la faisabilité vient de `faisabilite_indicateurs.csv` (`scripts/faisabilite_04.py`), les contrôles de `controles_coherence.csv` et `jointures.csv` (R-19). Aucune lecture causale.

**Ce qu'on regarde** : pour chaque préfecture, quelles dimensions de la priorisation sont disponibles. La dimension risque est « non déterminable » partout : les accidents sont nationaux (R-10).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from IPython.display import display
from style_04 import *
appliquer_style()
pd.set_option("display.max_colwidth", 90)

## Faisabilité des 4 indicateurs

In [2]:
display(faisabilite("O5"))

,Indicateur,Prévu (03),Résultat (04),Maille obtenue,Période,Preuve (04),Réserve
ID,,,,,,,
O5-01,Nombre de déficits par territoire,Repli : sans la dimension risque,Repli,Préfecture,2022,C,"Recommandation C : vérification, pas investissement (R-17)"
O5-02,Rang de priorité,Repli : sans la dimension risque,Repli,Préfecture,2022,C,Recommandation C (R-17)
O5-03,Population concernée,Cible,Cible,Préfecture,2022,A,
O5-04,Écart à la cible,Repli : sans la dimension risque,Repli,Préfecture,2022,C,Recommandation C (R-17)


## Dimensions disponibles par préfecture

Une case vide vaut « non disponible », jamais 0 (R-10). Niveau de preuve entre parenthèses.

In [3]:
pref = gpd.read_file(INTERIM / "prefectures_2022.geojson")
livret = pd.read_csv(INTERIM / "livret02_population_2022.csv")
pop = livret[(livret["Niveau"] == "préfecture") & (livret["Groupe d’âges"] == "Total") & (livret["Milieu"] == "Total") & (livret["Sexe"] == "Ensemble")]
ref = pd.read_csv(INTERIM.parent / "reference" / "referentiel_prefectures.csv", dtype=str)
pop = pop.merge(ref, left_on="Unité", right_on="Nom Livret 02 (D7)").set_index("Préfecture")["Valeur"]
rp = pd.read_csv(INTERIM / "routes_par_prefecture.csv")
r = pd.read_csv(INTERIM / "etat_troncons_rattaches.csv")
lies = {n for liste in r.loc[r["Méthode"] != "non rattaché", "Noms du tracé"].dropna() for n in liste.split(" | ")}
ae = pd.read_csv(INTERIM / "auto_ecoles_prefecture.csv")
tableau = pd.DataFrame(index=sorted(pref["Préfecture"]))
tableau["Population 2022 (A)"] = pop.reindex(tableau.index).map(lambda v: "oui" if pd.notna(v) else "")
tableau["Risque (accidents)"] = ""
tableau["État du réseau évalué (C)"] = tableau.index.map(lambda p: "oui" if rp[(rp["Préfecture"] == p) & rp["route_nom"].isin(lies)]["km"].sum() > 0 else "")
tableau["Auto-écoles agréées (C)"] = tableau.index.map(lambda p: "oui" if ae[(ae["Préfecture"] == p) & ae["agregation"].isin(["Agréée", "Antenne agréée"])].shape[0] > 0 else "aucune")
tableau["Zone"] = pref.set_index("Préfecture")["Zone"].reindex(tableau.index)
display(tableau)
display(tableau.drop(columns="Zone").apply(lambda s: (s == "oui").sum()).to_frame("préfectures (sur 39)"))

,Population 2022 (A),Risque (accidents),État du réseau évalué (C),Auto-écoles agréées (C),Zone
Agou,oui,,oui,aucune,Plateaux
Agoè-Nyivé,oui,,oui,oui,Grand Lomé
Akébou,oui,,oui,aucune,Plateaux
Amou,oui,,oui,aucune,Plateaux
Anié,oui,,oui,oui,Plateaux
Assoli,oui,,oui,oui,Kara
Avé,oui,,oui,aucune,Maritime hors Grand Lomé
Bas-Mono,oui,,oui,aucune,Maritime hors Grand Lomé
Bassar,oui,,oui,aucune,Kara
Binah,oui,,oui,aucune,Kara


,préfectures (sur 39)
Population 2022 (A),39
Risque (accidents),0
État du réseau évalué (C),38
Auto-écoles agréées (C),16


**Lecture** : la priorisation (O5-01, O5-02, O5-04) repose sur deux dimensions en C : l'état du réseau et l'offre d'auto-écoles. Une recommandation portée par du C déclenche une vérification, pas un investissement (R-17).